In [3]:
import pandas as pd

In [4]:
df = pd.read_parquet('../Data/cleaned_europe_energy.parquet')

df['country'] = df['country'].astype('category')

## Creating lagged variables (24h and 168h lags) and rolling variables to equip the ML model with "memory" about daily-weekly cycles and market trends that algorithms do not natively understand.

In [5]:
df = df.sort_values(by=['country', 'date']).reset_index(drop=True)

df['price_lag_24h'] = df.groupby('country')['price'].shift(24)
df['price_lag_168h'] = df.groupby('country')['price'].shift(168)
df['solar_lag_24h'] = df.groupby('country')['solar'].shift(24)
df['total_wind'] = df['wind_onshore'] + df['wind_offshore']
df['wind_lag_24h'] = df.groupby('country')['total_wind'].shift(24)

df['price_rolling_mean_24h'] = df.groupby('country')['price'].transform(
    lambda x: x.rolling(window=24, min_periods=1).mean()
)

df_final = df.dropna(subset=['price_lag_168h']).copy()

print(len(df))
print(len(df_final))
df_final.head()

301391
297527


,country,price,nuclear,fossil_gas,solar,waste,wind_offshore,wind_onshore,_hydro_pumped_storage_actual_consumption_,biomass,...,month,day_of_week,is_weekend,season,price_lag_24h,price_lag_168h,solar_lag_24h,total_wind,wind_lag_24h,price_rolling_mean_24h
168,Austria,82.07,NaN,923.00,0.0,100.0,NaN,3383.0,NaN,176.0,...,1,6,1,1,84.08,0.10,0.0,NaN,NaN,86.241667
169,Austria,81.09,NaN,948.25,0.0,100.0,NaN,3355.0,NaN,176.0,...,1,0,0,1,79.82,0.01,0.0,NaN,NaN,86.294583
170,Austria,72.68,NaN,981.25,0.0,100.0,NaN,3360.0,NaN,172.0,...,1,0,0,1,76.76,0.02,0.0,NaN,NaN,86.124583
171,Austria,69.96,NaN,1035.00,0.0,100.0,NaN,3365.0,NaN,175.0,...,1,0,0,1,73.46,0.00,0.0,NaN,NaN,85.978750
172,Austria,78.06,NaN,1121.75,0.0,100.0,NaN,3357.0,NaN,172.0,...,1,0,0,1,71.86,-0.01,0.0,NaN,NaN,86.237083


In [6]:
df_final.to_parquet('../Data/features_europe_energy.parquet', index=False)
